# CordiaAI — Step 15: Chest X-ray branch (NIH ChestX-ray14 + ViT-B/16)

Fine-tunes the same pretrained ViT-B/16 as the ECG branch to detect the three heart-related findings
on a chest X-ray: **Cardiomegaly** (enlarged heart), **Edema** (fluid in the lungs) and **Effusion**
(fluid around the lungs). Multi-label: one sigmoid per finding.

**Where to run:** a **Kaggle notebook** is easiest. `Add Data` → search *NIH Chest X-rays*
(`nih-chest-xrays/data`) and it is mounted under `/kaggle/input/`, so the 45 GB never has to be downloaded.
Settings → Accelerator → GPU T4. On Colab instead, run
`python download_imaging_datasets.py download nih_cxr --dest /content/nih` first (needs your kaggle.json).

**Split:** the dataset's official patient-level `train_val_list.txt` / `test_list.txt`, with 10% of the
train_val *patients* held out for validation, so no patient appears in two splits.

**Same code as the app:** the cells marked `%%writefile` are copies of this project's own modules
(`src/models/imaging/`, `src/imaging/`), generated from them, so the model trained here is exactly
the one `mobile_app/backend/main.py` loads.

In [ ]:
!pip install -q transformers scikit-learn
import torch
print('CUDA:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import os
for d in ['src/models/imaging', 'src/imaging']:
    os.makedirs(d, exist_ok=True)
for f in ['src/__init__.py', 'src/models/__init__.py', 'src/models/imaging/__init__.py', 'src/imaging/__init__.py']:
    open(f, 'a').close()

In [ ]:
%%writefile src/models/imaging/vit_backbone.py
"""Shared ViT-B/16 backbone for the imaging branches (X-ray, echo).

Same pretrained model the ECG branch uses (google/vit-base-patch16-224-in21k),
so every image-like branch in CordiaAI is a Vision Transformer.

Two details worth knowing:
- `pretrained=False` builds the same architecture with random weights. Local
  inference uses this, since the trained checkpoint overwrites every weight
  anyway -- it avoids downloading ~350MB of ImageNet weights at app startup.
- Layer freezing matches parameter names with a regex that accepts both
  "encoder.layer.N." and "layers.N.", since transformers versions differ
  (see the note in notebooks/step5_ecg_vit_colab.ipynb).
"""
from __future__ import annotations

import re

from torch import nn
from transformers import ViTConfig, ViTModel

VIT_NAME = "google/vit-base-patch16-224-in21k"
_LAYER_RE = re.compile(r"\blayers?\.(\d+)\.")


def load_vit(pretrained: bool = True, config: ViTConfig | None = None) -> ViTModel:
    if config is not None:
        return ViTModel(config)
    return ViTModel.from_pretrained(VIT_NAME) if pretrained else ViTModel(ViTConfig())


def freeze_below(vit: ViTModel, train_from_layer: int) -> None:
    """Train only encoder layers >= train_from_layer (plus nothing else in the ViT)."""
    for name, p in vit.named_parameters():
        match = _LAYER_RE.search(name)
        p.requires_grad = bool(match) and int(match.group(1)) >= train_from_layer


class Projection(nn.Sequential):
    def __init__(self, d_in: int, d_embed: int):
        super().__init__(nn.Linear(d_in, d_embed), nn.LayerNorm(d_embed))

In [ ]:
%%writefile src/models/imaging/cxr_vit.py
"""Chest X-ray branch: ViT-B/16 fine-tuned for heart-related findings.

Trained on NIH ChestX-ray14 (or CheXpert) in
`notebooks/step15_xray_vit_kaggle.ipynb`. Multi-label: an X-ray can show
several findings at once, so each finding has its own sigmoid output rather
than one softmax over classes.
"""
from __future__ import annotations

import torch
from torch import nn
from transformers import ViTConfig

from src.models.imaging.vit_backbone import Projection, freeze_below, load_vit

# The three ChestX-ray14 / CheXpert findings that are about the heart:
# an enlarged heart, fluid in the lungs (often from heart failure) and fluid
# around the lungs.
CXR_FINDINGS = ["Cardiomegaly", "Edema", "Effusion"]


class CXRViT(nn.Module):
    def __init__(
        self,
        n_findings: int = len(CXR_FINDINGS),
        d_embed: int = 256,
        train_from_layer: int = 8,
        pretrained: bool = True,
        config: ViTConfig | None = None,
    ):
        super().__init__()
        self.vit = load_vit(pretrained, config)
        freeze_below(self.vit, train_from_layer)
        self.project = Projection(self.vit.config.hidden_size, d_embed)
        self.classifier = nn.Linear(d_embed, n_findings)

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, 3, 224, 224) -> E_x, the 256-dim X-ray embedding."""
        return self.project(self.vit(pixel_values=x).last_hidden_state[:, 0])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns one logit per finding; apply sigmoid for probabilities."""
        return self.classifier(self.embed(x))

In [ ]:
%%writefile src/imaging/preprocess.py
"""Image preprocessing shared by training notebooks and the app.

Keeping one copy of each conversion matters for the same reason it did for
the ECG spectrogram (see gradcam_ecg.py): the app must feed a model exactly
what it was trained on, or predictions silently degrade.
"""
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

VIT_SIZE = 224


def _vit_normalise(x: np.ndarray) -> np.ndarray:
    # google/vit-base-patch16-224-in21k's preprocessor: scale to [0,1], then mean 0.5 / std 0.5
    return (x.astype(np.float32) / 255.0 - 0.5) / 0.5


# ---------------------------------------------------------------- X-ray
def xray_to_tensor(image: Image.Image) -> torch.Tensor:
    """Any chest X-ray image (PNG/JPG, grey or colour) -> (3, 224, 224)."""
    grey = image.convert("L").resize((VIT_SIZE, VIT_SIZE), Image.BILINEAR)
    arr = np.stack([np.array(grey)] * 3, axis=-1)
    return torch.from_numpy(_vit_normalise(arr)).permute(2, 0, 1)


# ---------------------------------------------------------------- Echo
def read_video_frames(path: str | Path) -> np.ndarray:
    """All frames of a video file as (frames, H, W) uint8 grey."""
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise ValueError(f"No frames could be read from {path}")
    return np.stack(frames)


def read_video_bytes(data: bytes, suffix: str = ".avi") -> np.ndarray:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / f"upload{suffix}"
        path.write_bytes(data)
        return read_video_frames(path)


def sample_clip(frames: np.ndarray, n_frames: int, stride: int = 2, start: int | None = None) -> np.ndarray:
    """n_frames frames, `stride` apart, starting at `start` (default: centred).
    Short videos are looped, as in the EchoNet-Dynamic reference code."""
    span = n_frames * stride
    if start is None:
        start = max(0, (len(frames) - span) // 2)
    idx = (start + np.arange(n_frames) * stride) % len(frames)
    return frames[idx]


def clip_to_tensor(clip: np.ndarray) -> torch.Tensor:
    """(frames, H, W) uint8 -> (frames, 3, 224, 224) normalised for the ViT."""
    t = torch.from_numpy(clip.astype(np.float32)).unsqueeze(1)
    t = F.interpolate(t, size=(VIT_SIZE, VIT_SIZE), mode="bilinear", align_corners=False)
    t = (t / 255.0 - 0.5) / 0.5
    return t.expand(-1, 3, -1, -1).contiguous()


# ---------------------------------------------------------------- MRI
# (CT has its own resampling pipeline in src/imaging/ct.py)
def normalise_volume(volume: np.ndarray, lo_pct: float = 1, hi_pct: float = 99) -> np.ndarray:
    """Percentile clip then scale to [0, 1] -- MRI intensities have no fixed unit."""
    lo, hi = np.percentile(volume, [lo_pct, hi_pct])
    return np.clip((volume.astype(np.float32) - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)


def slices_to_tensor(volume01: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) in [0,1] -> (slices, 1, size, size)."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(volume01, -1, 0))).unsqueeze(1)
    return F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)


@torch.no_grad()
def segment_mri_volume(model: torch.nn.Module, volume: np.ndarray, batch: int = 8) -> np.ndarray:
    """Short-axis MRI volume (H, W, slices) -> label mask of the same shape."""
    h, w, _ = volume.shape
    x = slices_to_tensor(normalise_volume(volume))
    labels = []
    for i in range(0, len(x), batch):
        logits = model(x[i : i + batch])
        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
        labels.append(logits.argmax(1))
    return np.moveaxis(torch.cat(labels).numpy().astype(np.uint8), 0, -1)


def mask_slices_to_tensor(mask: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) integer labels -> (slices, size, size), nearest-neighbour so labels stay whole."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(mask, -1, 0))).unsqueeze(1).float()
    return F.interpolate(t, size=(size, size), mode="nearest")[:, 0].long()

## Find the data and build labels

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from src.models.imaging.cxr_vit import CXR_FINDINGS

SEARCH_ROOTS = ['/kaggle/input', '/content/nih', 'nih']
entry = next(p for r in SEARCH_ROOTS for p in glob.glob(f'{r}/**/Data_Entry_2017*.csv', recursive=True))
data_root = os.path.dirname(entry)
print('Data root:', data_root)

paths = {os.path.basename(p): p for p in glob.glob(f'{data_root}/**/images*/**/*.png', recursive=True)}
df = pd.read_csv(entry)
df = df[df['Image Index'].isin(paths)].copy()
df['path'] = df['Image Index'].map(paths)
for f in CXR_FINDINGS:
    df[f] = df['Finding Labels'].str.split('|').apply(lambda labels: int(f in labels))
print(f'{len(df)} images with files found')
print(df[CXR_FINDINGS].sum())

def read_list(name):
    hits = glob.glob(f'{data_root}/**/{name}', recursive=True)
    return set(open(hits[0]).read().split()) if hits else None

test_names = read_list('test_list.txt')
if test_names is None:  # fall back to a patient-level 80/20 split
    rng = np.random.default_rng(42)
    pats = df['Patient ID'].unique(); rng.shuffle(pats)
    test_names = set(df[df['Patient ID'].isin(pats[: len(pats) // 5])]['Image Index'])
test_df = df[df['Image Index'].isin(test_names)]
trval = df[~df['Image Index'].isin(test_names)]
rng = np.random.default_rng(42)
pats = trval['Patient ID'].unique(); rng.shuffle(pats)
val_pats = set(pats[: len(pats) // 10])
val_df, train_df = trval[trval['Patient ID'].isin(val_pats)], trval[~trval['Patient ID'].isin(val_pats)]

# Keep every image with a heart finding plus a sample of the rest, so an epoch fits a free GPU session.
MAX_NEGATIVES = 30000
pos = train_df[train_df[CXR_FINDINGS].any(axis=1)]
neg = train_df[~train_df[CXR_FINDINGS].any(axis=1)].sample(n=min(MAX_NEGATIVES, (~train_df[CXR_FINDINGS].any(axis=1)).sum()), random_state=42)
train_df = pd.concat([pos, neg]).sample(frac=1, random_state=42)
print(f'Train {len(train_df)}  Val {len(val_df)}  Test {len(test_df)}')

## Dataset and model
No horizontal flips: the heart sits on the left, and flipping would teach the model the wrong anatomy.

In [ ]:
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.imaging.preprocess import xray_to_tensor
from src.models.imaging.cxr_vit import CXRViT

augment = transforms.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05))

class CXRDataset(Dataset):
    def __init__(self, frame, train):
        self.paths, self.y, self.train = frame['path'].tolist(), frame[CXR_FINDINGS].values.astype(np.float32), train
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i])
        if self.train:
            img = augment(img.convert('L'))
        return xray_to_tensor(img), self.y[i]

BATCH = 48
train_loader = DataLoader(CXRDataset(train_df, True), batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(CXRDataset(val_df, False), batch_size=96, num_workers=4)
test_loader = DataLoader(CXRDataset(test_df, False), batch_size=96, num_workers=4)

model = CXRViT().to(device)
print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Train (class-weighted BCE: each finding is rare, as HYP was for the ECG branch)

In [ ]:
from sklearn.metrics import roc_auc_score

EPOCHS = 3
pos_rate = train_df[CXR_FINDINGS].values.mean(0)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - pos_rate) / pos_rate, dtype=torch.float32, device=device))
optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-5, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

def predict(loader):
    model.eval(); probs, ys = [], []
    with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
        for xb, yb in loader:
            probs.append(torch.sigmoid(model(xb.to(device))).float().cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(ys)

best_auc, best_state = 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for step, (xb, yb) in enumerate(train_loader):
        with torch.autocast(device, enabled=device == 'cuda'):
            loss = criterion(model(xb.to(device)), yb.to(device))
        optimiser.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimiser); scaler.update()
        if step % 200 == 0:
            print(f'epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}')
    p, y = predict(val_loader)
    aucs = [roc_auc_score(y[:, k], p[:, k]) for k in range(len(CXR_FINDINGS))]
    print(f'Epoch {epoch} val AUROC:', {f: round(float(a), 3) for f, a in zip(CXR_FINDINGS, aucs)})
    if best_state is None or np.nanmean(aucs) > best_auc:
        best_auc, best_state = np.nanmean(aucs), {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

## Test (official test patients, never used for training or model selection)
Thresholds are picked on validation (Youden's J) and then applied unchanged to test.

In [ ]:
import json
from sklearn.metrics import f1_score, roc_curve

pv, yv = predict(val_loader)
thresholds = {}
for k, f in enumerate(CXR_FINDINGS):
    fpr, tpr, thr = roc_curve(yv[:, k], pv[:, k])
    thresholds[f] = float(thr[np.argmax(tpr - fpr)])
    if not np.isfinite(thresholds[f]):  # validation had no positives for this finding
        thresholds[f] = 0.5

pt, yt = predict(test_loader)
for k, f in enumerate(CXR_FINDINGS):
    pred = (pt[:, k] >= thresholds[f]).astype(int)
    print(f'{f:12s} AUROC {roc_auc_score(yt[:, k], pt[:, k]):.3f}  F1 {f1_score(yt[:, k], pred):.3f}  threshold {thresholds[f]:.3f}')

## Save and download
Put both files in `cordia_ai/checkpoints/`.

In [ ]:
torch.save(best_state, 'cxr_vit.pt')
json.dump(thresholds, open('cxr_thresholds.json', 'w'), indent=1)
import os
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:  # Kaggle also ships a google.colab module, so test for Kaggle itself
    from IPython.display import FileLink, display
    print('Download both (or use the Output panel, /kaggle/working) before stopping the session:')
    display(FileLink('cxr_vit.pt'), FileLink('cxr_thresholds.json'))
else:
    from google.colab import files
    files.download('cxr_vit.pt'); files.download('cxr_thresholds.json')